# MTSamples — Urology Subset EDA

Filtering to `medical_specialty == "Urology"`

In [1]:
!pip install -q textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import re
import textstat

pd.set_option("display.max_colwidth", 80)

CSV_URL = "https://raw.githubusercontent.com/socd06/medical-nlp/master/data/mtsamples.csv"
full_df = pd.read_csv(CSV_URL)

df = full_df[full_df["medical_specialty"].str.strip().str.lower() == "urology"].copy()
print(f"Urology samples: {len(df)} out of {len(full_df)} total")
df.head()

Urology samples: 158 out of 4999 total


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
18,18,Fertile male with completed family. Elective male sterilization via bilate...,Urology,Vasectomy - 4,"PROCEDURE: , Elective male sterilization via bilateral vasectomy.,PREOPERATI...","urology, sterilization, vas, fertile male, bilateral vasectomy, vasectomy, c..."
20,20,Whole body radionuclide bone scan due to prostate cancer.,Urology,Whole Body Radionuclide Bone Scan,"INDICATION:, Prostate Cancer.,TECHNIQUE:, 3.5 hours following the intraven...","urology, prostate cancer, technetium, whole body, urinary retention, bone sc..."
22,22,Normal vasectomy,Urology,Vasectomy - 1,"DESCRIPTION:, The patient was placed in the supine position and was prepped...","urology, vasectomy, allis clamp, catgut, hemoclips, iris scissors, scrotal, ..."
23,23,Voluntary sterility. Bilateral vasectomy. The vas deferens was grasped wi...,Urology,Vasectomy,"PREOPERATIVE DIAGNOSIS: , Voluntary sterility.,POSTOPERATIVE DIAGNOSIS: , Vo...","urology, hemiscrotum, bilateral vasectomy, voluntary sterility, vas deferens..."
24,24,Blood in urine - Transitional cell cancer of the bladder.,Urology,Urology Consut - 1,"CHIEF COMPLAINT:,",NaN


## 1. Sample names in this subset

In [3]:
df["sample_name"].value_counts().head(20)

,count
sample_name,
Vasectomy - 4,1
Whole Body Radionuclide Bone Scan,1
Vasectomy - 1,1
Vasectomy,1
Urology Consut - 1,1
Vasectomy - 2,1
Urology Discharge Summary,1
Umbilical Hernia Repair,1
Urology Letter,1


## 2. Missing / empty values check

In [4]:
print("Missing values per column:")
print(df.isna().sum())
print()
empty_transcription = df["transcription"].fillna("").str.strip() == ""
print(f"Empty transcriptions: {empty_transcription.sum()}")

Missing values per column:
Unnamed: 0            0
description           0
medical_specialty     0
sample_name           0
transcription         2
keywords             16
dtype: int64

Empty transcriptions: 2


## 3. Duplicate transcription check

In [5]:
n_duplicates = df["transcription"].duplicated().sum()
print(f"Exact duplicate transcriptions: {n_duplicates} out of {len(df)}")
df[df["transcription"].duplicated(keep=False)][["sample_name", "transcription"]].head(10)

Exact duplicate transcriptions: 1 out of 158


,sample_name,transcription
97,Inguinal orchiopexy,NaN
116,Hydrocele Repair,NaN


## 4. Transcription length

In [6]:
df["word_count"] = df["transcription"].fillna("").apply(lambda t: len(str(t).split()))
df["char_count"] = df["transcription"].fillna("").apply(lambda t: len(str(t)))
df["sentence_count"] = df["transcription"].fillna("").apply(
    lambda t: len(re.split(r'[.!?]+', t.strip())) if t.strip() else 0
)

df[["word_count", "char_count", "sentence_count"]].describe()

,word_count,char_count,sentence_count
count,158.000000,158.000000,158.000000
mean,401.386076,2605.113924,35.968354
std,293.154005,1838.802521,22.836562
min,0.000000,0.000000,0.000000
25%,201.250000,1339.250000,19.000000
50%,334.500000,2230.500000,32.500000
75%,529.750000,3516.250000,45.750000
max,2008.000000,12574.000000,138.000000


## 5. Readability — urology subset vs. full corpus

In [7]:
def safe_readability(text):
    text = str(text)
    if len(text.split()) < 10:
        return pd.Series({"flesch_reading_ease": None, "flesch_kincaid_grade": None})
    return pd.Series({
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
    })

readability = df["transcription"].fillna("").apply(safe_readability)
df = pd.concat([df, readability], axis=1)

urology_stats = df[["flesch_reading_ease", "flesch_kincaid_grade"]].mean()

full_readability = full_df["transcription"].fillna("").apply(safe_readability)
full_stats = full_readability.mean()

pd.DataFrame({"Urology": urology_stats, "All specialties": full_stats})

,Urology,All specialties
flesch_reading_ease,46.201310,44.536200
flesch_kincaid_grade,9.979913,10.106561


## 6. Correlation between length and readability
Does a longer urology report tend to be harder to read, within this subset?

In [8]:
df[["word_count", "sentence_count", "flesch_kincaid_grade", "flesch_reading_ease"]].corr()

,word_count,sentence_count,flesch_kincaid_grade,flesch_reading_ease
word_count,1.000000,0.836589,-0.034588,0.223018
sentence_count,0.836589,1.000000,-0.177278,0.200421
flesch_kincaid_grade,-0.034588,-0.177278,1.000000,-0.924506
flesch_reading_ease,0.223018,0.200421,-0.924506,1.000000


## 7. Most common keywords in urology samples

In [9]:
all_keywords = df["keywords"].dropna().str.split(",").explode().str.strip().str.lower()
all_keywords.value_counts().head(20).to_frame("count")

,count
keywords,
urology,141
,93
bladder,32
cystoscopy,23
inguinal,20
prostate,19
hernia,18
: thesetranscribed medical transcription sample reports and examples are provided by various users andare for reference purpose only. mthelpline does not certify accuracy and quality of sample reports.these transcribed medical transcription sample reports may include some uncommon or unusual formats;this would be due to the preference of the dictating physician. all names and dates have beenchanged (or removed) to keep confidentiality. any resemblance of any type of name or date orplace or anything else to real world is purely incidental.,18
circumcision,17


## 8. Procedure-type grouping
Rough grouping by keyword found in the sample name, to see which urology procedures are best represented.

In [10]:
procedure_patterns = {
    "Vasectomy / sterilization": r"vasectom|sterilization",
    "Prostate (prostatectomy/TURP/biopsy)": r"prostat|turp",
    "Bladder (cystectomy/TURBT/cystoscopy)": r"bladder|cyst(ectomy|oscopy)|turbt",
    "Kidney / renal / stone": r"kidney|renal|nephro|calculus|stone",
    "Scrotal / testicular": r"scrotal|testic|spermatocele|hydrocele|orchidopexy",
    "Circumcision": r"circumcision",
    "Incontinence / sling": r"incontinen|sling",
}

def match_procedure(name):
    name = str(name).lower()
    for label, pattern in procedure_patterns.items():
        if re.search(pattern, name):
            return label
    return "Other / unmatched"

df["procedure_group"] = df["sample_name"].apply(match_procedure)
df["procedure_group"].value_counts()

,count
procedure_group,
Other / unmatched,81
Prostate (prostatectomy/TURP/biopsy),23
Bladder (cystectomy/TURBT/cystoscopy),18
Circumcision,14
Scrotal / testicular,11
Vasectomy / sterilization,6
Kidney / renal / stone,3
Incontinence / sling,2


## 9. Readability by procedure group

In [11]:
df.groupby("procedure_group")[["word_count", "flesch_kincaid_grade"]].agg(["mean", "count"])

word_count       flesch_kincaid_grade  \
                                             mean count                 mean   
procedure_group                                                                
Bladder (cystectomy/TURBT/cystoscopy)  380.611111    18            10.791282   
Circumcision                           223.428571    14             8.824199   
Incontinence / sling                   418.500000     2             9.738027   
Kidney / renal / stone                 335.666667     3            10.709419   
Other / unmatched                      427.530864    81             9.885851   
Prostate (prostatectomy/TURP/biopsy)   501.782609    23            10.954287   
Scrotal / testicular                   358.909091    11             9.016114   
Vasectomy / sterilization              246.166667     6             9.068073   

                                             
                                      count  
procedure_group                              
Bladder (cystectomy/TURBT/cystoscopy)    18  
Circumcision                             14  
Incontinence / sling                      2  
Kidney / renal / stone                    3  
Other / unmatched                        79  
Prostate (prostatectomy/TURP/biopsy)     23  
Scrotal / testicular                     10  
Vasectomy / sterilization                 6